In [ ]:
# ============================================================
# CONFIGURAZIONE AMBIENTE
# ============================================================

import os
from pathlib import Path

REPO_DIR = Path("/content/Progetto-AI-VacuumCleaner")

if not REPO_DIR.exists():
    !git clone https://github.com/massafdc/Progetto-AI-VacuumCleaner.git

%cd /content/Progetto-AI-VacuumCleaner

!pip install -q -r requisiti.txt


# Progetto Vacuum-Cleaner

Introduzione all'IA, Informatica 2025/2026<br>
Massa Giorgio <br>
Stamponi Giovanni

# Introduzione
Il progetto Vacuum Cleaner riguarda lo sviluppo di un sistema che simula il comportamento di un robot aspirapolvere che si adatta all'ambiente. <br> L'obiettivo del progetto è integrare tecniche di elaborazione delle immagini, classificatori, algoritmi di ricerca informata e non, al fine di permettere al robot di interpretare l'ambiente e scegliere le azioni più appropriate. <br> Il sistema è composto da diversi moduli, che vengono integrati per ottenere una simulazione completa del comportamento del robot, dall'analisi dell'immagine della foto fino alla pianificazione dei suoi movimenti.

# 1 - Descrizione formale del dominio <br>
**1.1 - Rappresentazione dello stato**

L'ambiente è rappresentato da una griglia quadrata, in cui ogni cella può assumere i seguenti valori: S (Inizio), F (Fine), C (pulita), D (sporca)  V (molto sporca), X (non accessibile).

Gli stati sono rappresentati come S = ( P , G ) dove P rappresenta la posizione corrente del robot e G la configurazione attuale della griglia.

**1.2 Azioni del robot:**

UP/DOWN/LEFT/RIGHT: il robot esegue un movimento verso su/giù/sinistra/destra.
<br>
CLEAN: il robot pulisce la casella su cui si trova.


**1.3 Vincoli e assunzioni:**

*   Nella griglia devono essere presenti solo una cella S ed F ciascuna; queste sono considerate come celle pulite.
* Ogni azione ha costo uguale a 1.
* Il robot si può muovere solo all'interno della tabella: se ad esempio si trova nell'angolo in alto a sinistra, potrà spostarsi solo verso giù o verso destra.
* Quando il robot pulisce una casella D, questa diventa C; se pulisce una casella V, questa diventa D.

**1.4  Goal**
La condizione di goal è che il robot si trovi nella casella F e tutte le celle accessibili abbiano valore C. <br>

# 2 - Ricerca nello spazio degli stati
**2.1 Formulazione come problema di ricerca**

Il dominio è stato implementato come sottoclasse SmartVacuum della classe Problem di AIMA-python, ridefinendo i metodi actions, result, goal_test e path_cost secondo la formalizzazione descritta nel Capitolo 1. Ogni stato è rappresentato come una tupla (posizione, griglia), dove la griglia è a sua volta rappresentata come tupla di tuple, per garantirne l'immutabilità e la possibilità di essere utilizzata come chiave in insiemi e dizionari durante la ricerca

**2.2 Euristica**

Per guidare la ricerca informata (A*) è stata implementata un'euristica basata su una **Minimum Spanning Tree (MST)**, definita come:

$$
h(n) = costo\ minimo\ di\ pulizia\ rimanente + costo\ MST
$$

Il costo minimo di pulizia rimanente viene calcolato sommando 1 per ogni cella *D* ancora presente nella griglia e 2 per ogni cella `V`, poiché una cella *V* richiede due azioni *CLEAN*.

Per il calcolo del costo MST vengono considerati come nodi la posizione corrente del robot, tutte le celle ancora sporche e la posizione finale (goal). Il peso di ogni collegamento tra due nodi è dato dalla relativa **distanza di Manhattan**. La MST rappresenta quindi il costo minimo necessario, in un ambiente privo di ostacoli, per collegare tra loro tutti i punti che devono essere raggiunti.

L'euristica è **ammissibile**, in quanto nessuna delle due componenti sovrastima il costo reale rimanente. Il costo di pulizia rappresenta infatti il numero minimo di azioni *CLEAN* che devono necessariamente essere eseguite. Analogamente, il costo della MST costituisce un limite inferiore sul movimento necessario: la distanza di Manhattan ignora gli ostacoli *X* e quindi può solo sottostimare, o al massimo uguagliare, la distanza effettivamente percorribile dal robot. Inoltre, la MST individua il costo minimo per collegare la posizione del robot, tutte le celle sporche e il goal; qualsiasi soluzione valida deve necessariamente raggiungere tali punti.
Di conseguenza, la somma delle due componenti non può superare il costo della soluzione ottima rimanente:

$$
h(n) \leq h^*(n)
$$

e pertanto l'euristica risulta ammissibile per la ricerca A*.

**2.3 Ricerca informata  non informata**

Per verificare l'efficacia dell'euristica proposta, è stato implementato uno script di confronto **(test_search_performance.py)** che esegue sulla stessa istanza del problema sia la ricerca in ampiezza (Breadth-First Search, non informata) sia A* (informata), misurando il numero di nodi espansi, il tempo di esecuzione e il costo della soluzione trovata. Per contare i nodi espansi è stato introdotto un contatore (nodes_expanded), incrementato ad ogni chiamata del metodo actions.

In [ ]:
# esempio con tabella 3x3
!PYTHONPATH=. python tests/test_search_performance.py

## 3 - Percezione: acquisizione e classificazione delle immagini

**3.1 Dataset**

Per la classificazione delle lettere maiuscole C, D, F, S, V, X sono stati utilizzati due dataset complementari:

* EMNIST ByClass: dataset di lettere e cifre manoscritte, da cui sono state estratte esclusivamente le sei classi di interesse;
* Dataset digitale: immagini generate automaticamente a partire da diversi font di sistema (sia macOS sia Windows, per garantire la riproducibilità su entrambi i sistemi operativi utilizzati dai due membri del gruppo).

I due dataset sono stati combinati in un unico insieme di training e validation, mentre l'insieme di test è stato mantenuto composto esclusivamente da immagini EMNIST non viste durante l'addestramento. Questa scelta consente di valutare in modo onesto la capacità del classificatore di generalizzare su scrittura manoscritta reale, il caso più rappresentativo e più difficile fra i due.

**3.2 Architettura del classificatore**

Il classificatore adottato è una rete neurale MLP (Multi-Layer Perceptron, MLPClassifier di scikit-learn), con due strati nascosti di 256 e 128 neuroni rispettivamente, funzione di attivazione ReLU, ottimizzatore Adam ed early stopping basato su una porzione di validazione, per prevenire l'overfitting durante l'addestramento.

In [ ]:
!PYTHONPATH=. python -m src.classificatore.predict tests/immagini_lettere/PHOTO-2026-09-09-19-16-14.jpg

Caricamento modello...
Immagine utilizzata: /content/Progetto-AI-VacuumCleaner/tests/immagini_lettere/PHOTO-2026-09-09-19-16-14.jpg
Dimensioni originali: 1500 x 2000

--------------------------------
Predizione: X
Confidenza: 99.57%
--------------------------------
Figure(1600x400)

Statistiche immagine finale:
min:  0.0000
max:  0.6118
mean: 0.0424

Probabilità:
C: 0.07%
D: 0.00%
F: 0.05%
S: 0.02%
V: 0.28%
X: 99.57%


**3.3 Segmentazione della griglia dall'immagine**

Prima di classificare le singole lettere, è necessario individuare la griglia all'interno della fotografia ed estrarne le celle. Questa fase è implementata in **table_extractor.py** e si basa su OpenCV.

L'immagine viene inizialmente convertita in scala di grigi e sottoposta a una sogliatura adattiva, così da ottenere una rappresentazione binaria robusta anche in presenza di illuminazione non uniforme. Successivamente, mediante operazioni morfologiche di apertura con kernel rettangolari orientati orizzontalmente e verticalmente, vengono isolate rispettivamente le linee orizzontali e verticali della griglia.

Quando possibile, viene inoltre individuato il quadrilatero esterno della tabella. I quattro vertici vengono utilizzati per applicare una trasformazione prospettica, ottenendo una rappresentazione raddrizzata e quadrata della griglia. Questo permette di rendere più uniforme la successiva individuazione delle celle anche quando la fotografia è stata scattata da un'angolazione non perfettamente frontale.

Le linee vengono individuate analizzando le proiezioni dei pixel delle immagini contenenti le componenti orizzontali e verticali. Le coordinate che superano una determinata soglia vengono raggruppate mediante un clustering basato sulla distanza, così da ricondurre i pixel appartenenti a una stessa linea a un'unica posizione.

Nel caso in cui alcune linee esterne non vengano rilevate correttamente, vengono utilizzati come fallback i bordi dell'immagine raddrizzata. La griglia viene quindi validata verificando che presenti almeno una cella e, trattandosi di una griglia obbligatoriamente quadrata, che il numero di righe coincida con quello delle colonne.

Infine, per ogni intervallo tra due linee consecutive viene estratta una cella. Viene applicato un margine interno dell'8% per ridurre la presenza delle linee della griglia nell'immagine destinata al classificatore. Le celle ottenute vengono salvate singolarmente nella cartella `cells/`, mantenendo l'ordine per riga e colonna.


In [ ]:
!PYTHONPATH=. python -m src.classificatore.table_extractor tests/tabelle/table_test.png

Immagine caricata.
Cerco la tabella...
Tabella trovata.
Correggo la prospettiva...
Cerco le linee della griglia...
Linee orizzontali trovate: 4
Linee verticali trovate: 4
Griglia rilevata: 3 x 3
Estratte 9 celle.

Operazione completata.
Output: cells/


## 4 - Simulazione ed esecuzione completa
**4.1 Simulazione**

Una volta ottenuta la sequenza di azioni che risolve il problema, il modulo simulatore.py esegue il piano passo per passo, riutilizzando direttamente il metodo result() già definito in SmartVacuum, e produce un'immagine per ciascuno stato attraversato, oltre a un'animazione GIF dell'intera esecuzione.

Ogni fotogramma rappresenta la griglia con una colorazione diversa per ciascuno stato delle celle (bianco per le celle pulite, giallo per le celle sporche, arancione per le celle molto sporche), una "X" bianca sulle celle non accessibili (colorate di scuro), un'etichetta "START" sulla cella di partenza e "FINISH" sulla cella di arrivo (quest'ultima evidenziata anche da un bordo), e infine un'icona che rappresenta il robot nella sua posizione corrente.

**4.2 Pipeline di esecuzione**

L'intera pipeline di esecuzione è stata inserita nel programma **predict_table.py**
Questa sfrutta quindi tutti i moduli precedentemente descritti per restituire un quadro completo dell'esecuzione e la soluzione, anche simulata come gif, della tabella di cui viene passato come parametro il percorso. Inoltre è possibile decidere di risolvere il problema con solo algoritmo BFS, A* o entrambi, passando come parametro seguente rispettivamente 1, 2 o 3.

In [ ]:
!PYTHONPATH=. python -m src.classificatore.predict_table tests/tabelle/t_test5.png 3

Immagine caricata.
Cerco la tabella...
Tabella trovata.
Correggo la prospettiva...
Cerco le linee della griglia...
Linee orizzontali trovate: 5
Linee verticali trovate: 5
Griglia rilevata: 4 x 4
Estratte 16 celle.

=== CLASSIFICAZIONE CELLE ===

cell_00_00.png: D (100.00%)
cell_00_01.png: X (100.00%)
cell_00_02.png: X (100.00%)
cell_00_03.png: D (99.75%)
cell_01_00.png: V (83.07%)
cell_01_01.png: D (99.90%)
cell_01_02.png: D (99.90%)
cell_01_03.png: D (99.99%)
cell_02_00.png: D (99.20%)
cell_02_01.png: F (100.00%)
cell_02_02.png: X (75.97%)
cell_02_03.png: C (99.99%)
cell_03_00.png: D (99.59%)
cell_03_01.png: C (99.99%)
cell_03_02.png: S (100.00%)
cell_03_03.png: D (99.95%)

=== TABELLA RICONOSCIUTA ===

D X X D
V D D D
D F X C
D C S D

=== STATO SMART VACUUM ===

Start: (3, 2)
Goal:  (2, 1)

Griglia:
D X X D
V D D D
D C X C
D C C D

--- BFS (ricerca non informata) ---
Nodi espansi: 9966
Tempo: 2.055001 s
Lunghezza soluzione: 24
Costo soluzione: 24

Azioni:
1. RIGHT
2. CLEAN
3. UP
4. U

# Conclusioni

Il progetto ha permesso di realizzare una pipeline completa che integra percezione, ragionamento ed esecuzione.

I risultati sperimentali confermano che l'euristica adottata guida efficacemente la ricerca A*, riducendo significativamente il numero di nodi espansi rispetto a BFS a parità di soluzione ottima trovata.

La fase di percezione ha inoltre evidenziato come il passaggio da dati controllati a immagini reali introduca difficoltà legate principalmente a illuminazione, prospettiva, qualità dell'immagine e variabilità della scrittura. Nonostante ciò, le tecniche di preprocessing e segmentazione sviluppate hanno permesso di ottenere una pipeline sufficientemente robusta da collegare in modo automatico la visione del mondo reale alla componente di pianificazione.

Nel complesso, il progetto mostra come tecniche di visione artificiale, machine learning e ricerca nello spazio degli stati possano essere integrate all'interno di un unico sistema, nel quale ogni componente contribuisce alla realizzazione del comportamento finale del robot. Un possibile sviluppo futuro consiste nel migliorare ulteriormente la robustezza del riconoscimento delle celle e nell'estendere il sistema a griglie e scenari più complessi.
